# Notebook 04 — Full Pipeline Evaluation

Evaluates the complete co-design pipeline including:
- Syndrome-frequency sweep (Fig. 5)
- Latency vs. success improvement curves (Fig. 6)
- Retention–success Pareto frontier (Fig. 7)
- Qubit-count and depth scaling (Fig. 9)
- Overhead and runtime decomposition (Fig. 10)
- ILP kernel sensitivity (Table 6)

**Prerequisites:** Notebooks 00, 02, and 03 must be complete.

In [ ]:
import sys, os, json, time, warnings, pickle
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
from pathlib import Path

from qiskit import QuantumCircuit, transpile
from qiskit.qasm3 import loads as qasm3_loads
from qiskit.providers.fake_provider import GenericBackendV2
from qiskit.circuit.random import random_circuit

from src.compiler.mapping_pass import HardwareAwareMappingPass
from src.scheduler.scheduler import QEDScheduler
from src.scheduler.feature_extractor import CircuitFeatureExtractor
from src.qed.circuit_builder import QEDCircuitBuilder
from src.simulation.simulator import DensityMatrixSimulator
from src.simulation.noise_models import load_noise_model
from src.metrics.aggregator import MetricsAggregator
from src.metrics.statistical_tests import wilcoxon_test, bootstrap_ci
from src.utils.io import load_json

SEED = 42
np.random.seed(SEED)
DATA_DIR    = Path('../data')
RESULTS_DIR = DATA_DIR / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Load noise models
nm_sc  = load_noise_model('../noise_models/ibm_lagos.json')
nm_ti  = load_noise_model('../noise_models/trapped_ion.json')
nm_adv = load_noise_model('../noise_models/adversarial.json')
device_sc  = load_json('../noise_models/ibm_lagos.json')
device_ti  = load_json('../noise_models/trapped_ion.json')

# Load trained scheduler
with open(DATA_DIR / 'training' / 'model_checkpoint.pkl', 'rb') as f:
    ckpt = pickle.load(f)
scheduler = QEDScheduler(model_ds=ckpt['model_ds'], model_r=ckpt['model_r'],
                         lambda_param=0.8, mu_param=5e-4)

# Load benchmark circuits
def load_qasm(path):
    with open(path) as f:
        return qasm3_loads(f.read())

vqe_h2    = load_qasm('../circuits/vqe_h2.qasm')
vqe_lih   = load_qasm('../circuits/vqe_lih.qasm')
qpe_12    = load_qasm('../circuits/phase_est_12.qasm')
grover_10 = load_qasm('../circuits/grover_10.qasm')

simulator = DensityMatrixSimulator(use_gpu=False)
qed_builder = QEDCircuitBuilder()

print('Setup complete.')

## 1. Syndrome-Frequency Sweep (reproduces Fig. 5)

In [ ]:
FREQ_GRID = [0.0, 0.25, 0.50, 0.75, 1.0]
N_TRIALS  = 50   # increase to 100 for publication

freq_results = {profile: {f: [] for f in FREQ_GRID}
                for profile in ['Superconducting', 'Trapped-ion', 'Adversarial']}

noise_map = {
    'Superconducting': (nm_sc, device_sc),
    'Trapped-ion':     (nm_ti, device_ti),
    'Adversarial':     (load_noise_model('../noise_models/adversarial.json'),
                        load_json('../noise_models/adversarial.json')),
}

mapper_cfg = dict(T0=1.0, alpha=0.995, n_iter=5000, ilp_window=10,
                  latency_budget_ms=2000, latency_penalty=1.0)

for profile, (nm, dev) in noise_map.items():
    mapper = HardwareAwareMappingPass(device_props=dev, seed=SEED, **mapper_cfg)
    compiled_base, _ = mapper.run(vqe_h2)
    print(f'\n{profile}')
    for f in FREQ_GRID:
        s_list = []
        for t in range(N_TRIALS):
            if f == 0.0:
                circ_eval = compiled_base
                ret = 1.0
            else:
                circ_eval, ret = qed_builder.insert(
                    compiled_base, freq=f, positions=[0]
                )
            res = simulator.run(circ_eval, noise_model=nm, shots=1024,
                                seed=SEED + t)
            s_list.append(res['success_prob'] * ret)
        freq_results[profile][f] = np.array(s_list)
        print(f'  f={f:.2f}  S={np.mean(s_list):.3f} ± {np.std(s_list):.3f}')

# Persist for plotting notebook
import pickle
with open(RESULTS_DIR / 'freq_sweep_results.pkl', 'wb') as fh:
    pickle.dump(freq_results, fh)
print('\n✓ Freq-sweep complete.')

## 2. Latency vs. Success Improvement (reproduces Fig. 6)

In [ ]:
QUBIT_COUNTS = [8, 12, 16]
N_ITER_GRID  = [100, 500, 1000, 2000, 3000, 5000]

latency_rows = []

for n_q in QUBIT_COUNTS:
    circ = random_circuit(n_q, depth=max(20, n_q * 3), seed=SEED)
    backend = GenericBackendV2(num_qubits=n_q)
    baseline_compiled = transpile(circ, backend=backend,
                                   optimization_level=2,
                                   seed_transpiler=SEED)
    base_res = simulator.run(baseline_compiled, noise_model=nm_sc,
                              shots=1024, seed=SEED)
    S_base = base_res['success_prob']

    for n_iter in N_ITER_GRID:
        t0 = time.perf_counter()
        mapper_iter = HardwareAwareMappingPass(
            device_props=device_sc, T0=1.0, alpha=0.995,
            n_iter=n_iter, ilp_window=10,
            latency_budget_ms=5000, latency_penalty=1.0,
            seed=SEED,
        )
        compiled_iter, _ = mapper_iter.run(circ)
        sched = scheduler.predict(compiled_iter, device_sc)
        circ_qed, ret = qed_builder.insert(
            compiled_iter, freq=sched['freq'], positions=sched['positions']
        )
        t_compile = (time.perf_counter() - t0) * 1000

        res = simulator.run(circ_qed, noise_model=nm_sc, shots=1024, seed=SEED)
        S_joint = res['success_prob'] * ret
        delta_S  = S_joint - S_base

        latency_rows.append({
            'n_qubits': n_q,
            'n_iter':   n_iter,
            'latency_ms': round(t_compile, 1),
            'delta_S':  round(delta_S, 4),
            'S_base':   round(S_base, 4),
            'S_joint':  round(S_joint, 4),
        })
        print(f'n_q={n_q}  n_iter={n_iter:5d}  '
              f'lat={t_compile:6.1f}ms  ΔS={delta_S:+.3f}')

df_latency = pd.DataFrame(latency_rows)
df_latency.to_csv(RESULTS_DIR / 'latency_vs_improvement.csv', index=False)
print('✓ Latency sweep complete.')

## 3. Retention–Success Pareto Frontier (reproduces Fig. 7)

In [ ]:
from src.metrics.aggregator import compute_pareto_frontier

FREQ_CANDS = [0.0, 0.25, 0.50, 0.75, 1.0]
POS_CANDS  = [0, 1, 2, 3]

pareto_rows = []

mapper = HardwareAwareMappingPass(device_props=device_sc, seed=SEED, **mapper_cfg)
compiled_vqeh2, _ = mapper.run(vqe_h2)

for f in FREQ_CANDS:
    for pos in POS_CANDS:
        if f == 0.0:
            c_eval, ret = compiled_vqeh2, 1.0
        else:
            c_eval, ret = qed_builder.insert(compiled_vqeh2, freq=f, positions=[pos])
        s_list = []
        for t in range(30):
            r = simulator.run(c_eval, noise_model=nm_sc, shots=1024, seed=SEED+t)
            s_list.append(r['success_prob'])
        pareto_rows.append({
            'freq': f, 'pos': pos,
            'S': np.mean(s_list), 'R': ret,
        })

df_pareto = pd.DataFrame(pareto_rows)
df_pareto.to_csv(RESULTS_DIR / 'pareto_data.csv', index=False)
print(f'✓ Pareto grid: {len(df_pareto)} configurations evaluated.')

# Identify Pareto-optimal points
pareto_front = compute_pareto_frontier(df_pareto, obj1='S', obj2='R')
print(f'Pareto-optimal configurations: {len(pareto_front)}')

## 4. Qubit-count and Depth Scaling (reproduces Fig. 9)

In [ ]:
QUBIT_RANGE = list(range(6, 21, 2))
DEPTH_RANGE = [10, 20, 40, 60, 80, 100, 120, 140, 160]

scaling_rows = []

# --- Qubit scaling ---
for n_q in QUBIT_RANGE:
    circ = random_circuit(n_q, depth=max(20, n_q * 3), seed=SEED)
    backend = GenericBackendV2(num_qubits=n_q)

    # Baseline
    base_c = transpile(circ, backend=backend, optimization_level=2,
                        seed_transpiler=SEED)
    base_res = simulator.run(base_c, noise_model=nm_sc, shots=512, seed=SEED)
    S_base = base_res['success_prob']

    # Joint
    mapper_q = HardwareAwareMappingPass(
        device_props=device_sc, seed=SEED,
        T0=1.0, alpha=0.995, n_iter=2000, ilp_window=10,
        latency_budget_ms=5000, latency_penalty=1.0,
    )
    comp_q, _ = mapper_q.run(circ)
    sched = scheduler.predict(comp_q, device_sc)
    c_qed, ret = qed_builder.insert(comp_q, freq=sched['freq'],
                                     positions=sched['positions'])
    joint_res = simulator.run(c_qed, noise_model=nm_sc, shots=512, seed=SEED)
    S_joint = joint_res['success_prob'] * ret

    scaling_rows.append({
        'sweep': 'qubit_count', 'x': n_q,
        'S_baseline': round(S_base, 4), 'S_joint': round(S_joint, 4),
    })
    print(f'n_q={n_q:2d}  S_base={S_base:.3f}  S_joint={S_joint:.3f}')

# --- Depth scaling ---
for depth in DEPTH_RANGE:
    circ = random_circuit(10, depth=depth, seed=SEED)
    backend = GenericBackendV2(num_qubits=10)

    base_c = transpile(circ, backend=backend, optimization_level=2,
                        seed_transpiler=SEED)
    base_res = simulator.run(base_c, noise_model=nm_sc, shots=512, seed=SEED)
    S_base = base_res['success_prob']

    mapper_d = HardwareAwareMappingPass(
        device_props=device_sc, seed=SEED,
        T0=1.0, alpha=0.995, n_iter=2000, ilp_window=10,
        latency_budget_ms=5000, latency_penalty=1.0,
    )
    comp_d, _ = mapper_d.run(circ)

    # ML-scheduled
    sched = scheduler.predict(comp_d, device_sc)
    c_ml, ret_ml = qed_builder.insert(comp_d, freq=sched['freq'],
                                       positions=sched['positions'])
    res_ml = simulator.run(c_ml, noise_model=nm_sc, shots=512, seed=SEED)
    S_ml = res_ml['success_prob'] * ret_ml

    # Uniform QED (f=0.5 fixed)
    c_uni, ret_uni = qed_builder.insert(comp_d, freq=0.5, positions=[0])
    res_uni = simulator.run(c_uni, noise_model=nm_sc, shots=512, seed=SEED)
    S_uni = res_uni['success_prob'] * ret_uni

    scaling_rows.append({
        'sweep': 'depth', 'x': depth,
        'S_baseline': round(S_base, 4), 'S_ml_qed': round(S_ml, 4),
        'S_uniform_qed': round(S_uni, 4),
    })
    print(f'depth={depth:3d}  S_base={S_base:.3f}  S_ml={S_ml:.3f}  S_uni={S_uni:.3f}')

df_scaling = pd.DataFrame(scaling_rows)
df_scaling.to_csv(RESULTS_DIR / 'scaling_results.csv', index=False)
print('\n✓ Scaling experiments complete.')

## 5. ILP Kernel Sensitivity (reproduces Table 6)

In [ ]:
ILP_WINDOWS = [5, 10, 15]
ilp_rows = []

for w in ILP_WINDOWS:
    s_list, lat_list = [], []
    for t in range(100):
        t0 = time.perf_counter()
        mapper_w = HardwareAwareMappingPass(
            device_props=device_sc,
            T0=1.0, alpha=0.995, n_iter=5000, ilp_window=w,
            latency_budget_ms=2000, latency_penalty=1.0,
            seed=SEED + t,
        )
        comp_w, _ = mapper_w.run(vqe_h2)
        sched = scheduler.predict(comp_w, device_sc)
        c_qed, ret = qed_builder.insert(comp_w, freq=sched['freq'],
                                         positions=sched['positions'])
        lat_ms = (time.perf_counter() - t0) * 1000
        res = simulator.run(c_qed, noise_model=nm_sc, shots=1024, seed=SEED + t)
        s_list.append(res['success_prob'] * ret)
        lat_list.append(lat_ms)

    s_arr   = np.array(s_list)
    ci      = bootstrap_ci(s_arr, n_boot=10_000, seed=SEED)
    ilp_rows.append({
        'w': w,
        'S_mean': round(s_arr.mean(), 3),
        'CI_lo':  round(ci[0], 3),
        'CI_hi':  round(ci[1], 3),
        'Latency_ms': round(np.median(lat_list), 0),
    })
    print(f'w={w:2d}  S={s_arr.mean():.3f} [{ci[0]:.3f},{ci[1]:.3f}]  '
          f'lat={np.median(lat_list):.0f}ms')

df_ilp = pd.DataFrame(ilp_rows)
print('\nTable 6 (ILP Kernel Sensitivity):')
print(df_ilp.to_string(index=False))
df_ilp.to_csv(RESULTS_DIR / 'ilp_sensitivity.csv', index=False)

## 6. Runtime Decomposition (reproduces Fig. 10b)

In [ ]:
circuit_map = {
    'VQE-H2': vqe_h2, 'VQE-LiH': vqe_lih,
    'QPE-12': qpe_12, 'Grover-10': grover_10,
}
runtime_rows = []

for cname, circ in circuit_map.items():
    t_compile_list, t_infer_list, t_sim_list, t_io_list = [], [], [], []
    for t in range(20):
        # Compile
        t0 = time.perf_counter()
        mapper_rt = HardwareAwareMappingPass(
            device_props=device_sc, seed=SEED + t,
            T0=1.0, alpha=0.995, n_iter=5000, ilp_window=10,
            latency_budget_ms=2000, latency_penalty=1.0,
        )
        comp_rt, _ = mapper_rt.run(circ)
        t_compile_list.append((time.perf_counter() - t0) * 1000)

        # ML inference
        t1 = time.perf_counter()
        sched = scheduler.predict(comp_rt, device_sc)
        t_infer_list.append((time.perf_counter() - t1) * 1000)

        # Simulation
        t2 = time.perf_counter()
        c_qed, ret = qed_builder.insert(comp_rt, freq=sched['freq'],
                                         positions=sched['positions'])
        res = simulator.run(c_qed, noise_model=nm_sc, shots=1024,
                             seed=SEED + t)
        t_sim_list.append((time.perf_counter() - t2) * 1000)
        t_io_list.append(2.0)  # nominal I/O cost

    runtime_rows.append({
        'Circuit': cname,
        'Compile_ms':  round(np.mean(t_compile_list), 1),
        'Inference_ms': round(np.mean(t_infer_list), 2),
        'Simulation_ms': round(np.mean(t_sim_list), 1),
        'IO_ms': round(np.mean(t_io_list), 1),
    })
    print(f'{cname:12s}  compile={np.mean(t_compile_list):.1f}ms  '
          f'infer={np.mean(t_infer_list):.1f}ms  '
          f'sim={np.mean(t_sim_list):.1f}ms')

df_runtime = pd.DataFrame(runtime_rows)
df_runtime.to_csv(RESULTS_DIR / 'runtime_breakdown.csv', index=False)
print('\n✓ Runtime decomposition complete.')